**Notebook_pour_imagerie_thermique.ipynb**

L. Halloran

Ceci est un notebook pour colab (https://colab.research.google.com/) pour vous aider à traiter vos données dans la première partie de l'activité "imagerie thermique". Si vous avez des questions, n'hésitez pas à en discuter avec l'encadrant..

ATTENTION: Sauvegardez vos modifications... ça ne se fait pas automatiquement. 😲


Importer les modules qu'il nous faut...

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from scipy.optimize import curve_fit

In [2]:
#Use to import the file into google Colab drive
from google.colab import files 
#Use to import io, which opens the file from the Colab drive
import io

ModuleNotFoundError: No module named 'google'

ici, vous allez uploader votre fichier...

In [ ]:
uploaded = files.upload()


maintenant, on convertit le fichier à un "data frame". On le fera pour chaque onglet de votre fichier excel...

In [ ]:
data1 = pd.read_excel('exemple_T.xlsx','x=10mm') # changez le nom du fichier et les noms des onglets/feuilles
data2 = pd.read_excel('exemple_T.xlsx','x=20mm')
# etc.


ceci efface le fichier uploadé (si vous voulez re-uploader un autre avec le même nom, par exemple)

In [ ]:
!rm {'exemple_T.xlsx'}

on verifie ce qu'on a uploadé...

In [ ]:
print(data1)
print(data2)
# etc.

pour selectioner les données qu'il vous faut pour l'ajustement de courbes...
il faut ecrire le nom exacte que vous avez donné aux colonnes de temperature et temps, e.g., `'temps (s)'`

In [ ]:
deltaT = data1['delta T (degC)'] # deltaT = T - T0
temps = data1['temps (s)']

# ici on enleve la première valeur (ça ne marchera pas avec l'équation car division par 0)...
deltaT = deltaT[1:]
temps = temps[1:]

Definir l'equation a ajuster. NOTEZ: changez la valeur de x (ici 0.01 m) pour correspondre avec les données selectionnées (e.g., x=10mm).

In [ ]:
def function_for_fit(t,alpha,c0): #eqn. 48, avec x = 10 mm. il faudra changer cet x pour chaque x different. n'oublie pas qu'on utilise des unitées SI (donc metres pas centimetres)
    return c0/(np.sqrt(4*np.pi*alpha*t) ) * np.exp( -0.01**2/(4*alpha*t) )

Voir si ce qu'on a fait a du sens. Ici on fait juste une figure avec des valeurs choisis au hazard pour vérifier que l'équation ci-dessus est bien.

In [ ]:
test_t = np.arange(600)
plt.plot(test_t,function_for_fit(test_t,1E-6,10)) # test plot avec alpha = 1E-6 m^2/s et c0 = 10 degC.

faire l'ajustement de courbe

In [ ]:
popt, pcov = curve_fit(function_for_fit, temps, deltaT)
#OPTIONEL...
#popt, pcov = curve_fit(function_for_fit, temps, deltaT, p0=[3E6,0.5]) #p0 c'est les valeurs initiales de l'ajustement. vous pouvez les changer si l'ajustement de courbe échoue.

print(popt)
print(pcov)

notre fonction a 2 parametres (alpha et c0). c'est alpha qui nous interesse. popt contient les valeurs, pvar contient les variances de ces valeurs.

In [ ]:
alpha_fit = popt[0]
alpha_ecart_type = np.sqrt(pcov[0,0])
c0_fit = popt[1]
c0_ecart_type = np.sqrt(pcov[1,1])
print('alpha = '+ str(alpha_fit))
print('alpha ecart type = '+ str(alpha_ecart_type))
print('c0 = '+ str(c0_fit))
print('c0 ecart type'+ str(c0_ecart_type))


Maintenant faites une figure de vos données et la courbe ajustée...

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.plot(temps,deltaT,'ko')
timeforgraph = np.arange(600)
ax.plot(function_for_fit(timeforgraph,alpha_fit,c0_fit),'r')
ax.set_xlabel('LABEL YOUR AXES!')

maintenant notez ces valeurs et refaire ces derniers etapes pour chaque distance x...